# Poster Figures (v2) — Larger, Poster-Friendly Versions

This notebook generates **poster-ready** visuals you can screenshot or export:

1) **Feature Set Ladder (full picture)** — big table + big PNG export
2) **Sharpe by Feature Set (Logistic, 3D)** — large (10×6)
3) **Sharpe across ALL configs** — ranked bar chart + highlight logistic+3D winners

Prereq: run the pipeline once so CSV outputs exist:
```bash
python -m src.pipeline --mode live --notebook 02
```


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd()
ROOT = HERE if (HERE / 'outputs').exists() else HERE.parent
OUT_METRICS = ROOT / 'outputs' / 'metrics'
OUT_FIG = ROOT / 'outputs' / 'figures'
OUT_FIG.mkdir(parents=True, exist_ok=True)

print('ROOT =', ROOT)
print('OUT_METRICS =', OUT_METRICS)
print('OUT_FIG =', OUT_FIG)


## 1) Feature Set Ladder (full picture)
This is the clearest way to show **what each feature set includes** (F1→F5).

In [ ]:
from src.config import AppConfig

config = AppConfig()
order = ['F1_momentum','F2_momentum_reversal','F3_plus_risk_liquidity','F4_plus_cross_sectional','F5_full_finance']

rows = []
for fs_name in order:
    groups = config.feature_sets.get(fs_name, [])
    rows.append({
        'Feature Set': fs_name,
        'What is included (feature groups)': ', '.join(groups),
    })
feature_ladder = pd.DataFrame(rows)
feature_ladder

In [ ]:
# Export a BIG readable PNG table for Canva
# Tip: if the second column is too long, Canva still handles it well when you scale the image.
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis('off')
tbl = ax.table(
    cellText=feature_ladder.values,
    colLabels=feature_ladder.columns,
    cellLoc='left',
    colLoc='left',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.6)
ax.set_title('Feature Set Ladder (F1 → F5): What is Included', fontsize=14, pad=14)
plt.tight_layout()
out_path = OUT_FIG / 'poster_feature_ladder_big.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 1B) Feature Group Map (full feature column list)
This produces the exact ‘full picture’ mapping you showed (group → list of included feature columns).
Screenshot the table or export to PNG for Canva.

Then we load CSV outputs from the pipeline run for Sharpe charts.

In [ ]:
from src.features import feature_group_map

fg = feature_group_map()
feature_map_df = pd.DataFrame([
    {'Feature Group': k, 'Included Feature Columns': ', '.join(v)}
    for k, v in fg.items()
])
feature_map_df

In [ ]:
# Export BIG PNG for Canva
fig, ax = plt.subplots(figsize=(14, 7))
ax.axis('off')
tbl = ax.table(
    cellText=feature_map_df.values,
    colLabels=feature_map_df.columns,
    cellLoc='left',
    colLoc='left',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.4)
ax.set_title('Feature Group Map: Full Included Feature Columns', fontsize=14, pad=14)
plt.tight_layout()
out_path = OUT_FIG / 'poster_feature_group_map_full.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

In [ ]:
metrics_path = OUT_METRICS / '02_live_metrics.csv'
backtest_path = OUT_METRICS / '02_live_backtest_summary.csv'

assert metrics_path.exists(), f'Missing: {metrics_path} (run pipeline first)'
assert backtest_path.exists(), f'Missing: {backtest_path} (run pipeline first)'

metrics = pd.read_csv(metrics_path)
backtest = pd.read_csv(backtest_path)

print('metrics:', metrics.shape)
print('backtest:', backtest.shape)
backtest.head(3)

## 2) Sharpe by feature set (Logistic, 3D) — big 10×6
This is your clean **feature engineering → economic metric** chart.

In [ ]:
sub = backtest[(backtest['model'] == 'logistic_regression') & (backtest['horizon_days'] == 3)].copy()
sub['__order'] = sub['feature_set'].apply(lambda x: order.index(x) if x in order else 999)
sub = sub.sort_values('__order').drop(columns='__order')

display(sub[['feature_set','sharpe','max_drawdown','annualized_return','annualized_volatility','avg_turnover','avg_cost_drag']])

plt.figure(figsize=(10, 6))
plt.bar(sub['feature_set'], sub['sharpe'])
plt.xticks(rotation=20, ha='right')
plt.ylabel('Sharpe')
plt.title('Sharpe by Feature Set (Logistic Regression, 3D)')
plt.tight_layout()
out_path = OUT_FIG / 'poster_sharpe_by_feature_set_logreg_3d_big.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

## 3) Sharpe across ALL configs (clear winner view)
Goal: show that **Logistic + 3D + deeper features** is best in this run.
We plot Sharpe for every (feature_set, model, horizon) and highlight Logistic+3D bars.

In [ ]:
all_bt = backtest.copy()
all_bt['label'] = all_bt.apply(lambda r: f"{r['feature_set']} | {r['model']} | {int(r['horizon_days'])}D", axis=1)
all_bt = all_bt.sort_values('sharpe', ascending=False).reset_index(drop=True)

# Color rule: highlight logistic+3D
colors = []
for _, r in all_bt.iterrows():
    if (r['model'] == 'logistic_regression') and (int(r['horizon_days']) == 3):
        colors.append('#2563eb')  # blue highlight
    else:
        colors.append('#9ca3af')  # gray

# Plot top N only to keep readable
N = min(12, len(all_bt))
top = all_bt.head(N).copy()
top_colors = colors[:N]

plt.figure(figsize=(10, 6))
plt.barh(top['label'][::-1], top['sharpe'][::-1], color=top_colors[::-1])
plt.xlabel('Sharpe')
plt.title('Top Sharpe Strategies (highlight: Logistic + 3D)')
plt.tight_layout()
out_path = OUT_FIG / 'poster_top_sharpe_all_configs_highlight.png'
plt.savefig(out_path, dpi=220)
plt.show()
print('Saved:', out_path)

display(top[['feature_set','model','horizon_days','sharpe','max_drawdown','annualized_return','avg_turnover','avg_cost_drag']])

## What to upload into Canva
- `outputs/figures/poster_feature_ladder_big.png`
- `outputs/figures/02_live_rankic_heatmap.png` (existing from pipeline)
- `outputs/figures/02_live_F5_full_finance_logistic_regression_3d_equity.png` (existing from pipeline)
- `outputs/figures/poster_sharpe_by_feature_set_logreg_3d_big.png`
- `outputs/figures/poster_top_sharpe_all_configs_highlight.png`
